# CloudWalk — Sourcing Analysis
**Technical Challenge: Patterns, Conversion & Recruiter Intelligence**

This notebook reproduces the full analysis built on a fictional sourcing dataset of 601 candidates across 6 recruiters, 8 source channels, and 4 seniority levels.

AI (Claude, Anthropic) was used as a thinking partner throughout — for pattern recognition, hypothesis stress-testing, and insight structuring.

---

## 0. Setup & Data Load

In [23]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Load dataset
# If running on Google Colab, upload the file first via Files panel on the left
df = pd.read_excel('mock_sourcing_dataset.xlsx')

print(f'Dataset loaded: {df.shape[0]} candidates, {df.shape[1]} columns')
print(f'Columns: {df.columns.tolist()}')

Dataset loaded: 601 candidates, 35 columns
Columns: ['candidate_id', 'candidate_name', 'candidate_email', 'requisition_id', 'role', 'department', 'location', 'work_mode', 'seniority', 'years_experience', 'hiring_manager', 'recruiter', 'source_channel', 'decision_source', 'sourced_date', 'first_contact_date', 'response_received', 'response_time_days', 'screening_pass', 'screening_date', 'interview1_pass', 'interview1_date', 'test_taken', 'test_date', 'technical_test_score', 'behavior_score', 'manager_score', 'offer_sent', 'offer_date', 'hired', 'hired_date', 'final_stage', 'rejection_reason', 'stage_duration_days', 'recruiter_notes']


In [24]:
from google.colab import files
uploaded = files.upload()

Saving mock_sourcing_dataset.xlsx to mock_sourcing_dataset (2).xlsx


## 1. Funnel Overview

Mapping the full conversion funnel from sourced to hired.

In [25]:
total = len(df)

funnel = {
    'Sourced':        total,
    'Responded':      int(df['response_received'].sum()),
    'Screening Pass': int(df['screening_pass'].sum()),
    'Interview Pass': int(df['interview1_pass'].sum()),
    'Test Taken':     int(df['test_taken'].sum()),
    'Offer Sent':     int(df['offer_sent'].sum()),
    'Hired':          int(df['hired'].sum())
}

funnel_df = pd.DataFrame({
    'Stage': funnel.keys(),
    'Count': funnel.values(),
    '% of Total': [f"{v/total*100:.1f}%" for v in funnel.values()]
})

print('=== CONVERSION FUNNEL ===')
print(funnel_df.to_string(index=False))

print(f'\nKey drop-off: Test Taken ({funnel["Test Taken"]}) → Offer Sent ({funnel["Offer Sent"]}) = {(1 - funnel["Offer Sent"]/funnel["Test Taken"])*100:.0f}% drop')

=== CONVERSION FUNNEL ===
         Stage  Count % of Total
       Sourced    601     100.0%
     Responded    408      67.9%
Screening Pass    308      51.2%
Interview Pass    234      38.9%
    Test Taken    201      33.4%
    Offer Sent     62      10.3%
         Hired     49       8.2%

Key drop-off: Test Taken (201) → Offer Sent (62) = 69% drop


## 2. Which Sourcing Strategies Convert Best?

Hire rate and response rate by source channel.

In [26]:
channels = df.groupby('source_channel').agg(
    Candidates=('candidate_id', 'count'),
    Responded=('response_received', 'sum'),
    Hired=('hired', 'sum')
).reset_index()

channels['Hire Rate'] = (channels['Hired'] / channels['Candidates'] * 100).round(1).astype(str) + '%'
channels['Response Rate'] = (channels['Responded'] / channels['Candidates'] * 100).round(1).astype(str) + '%'
channels = channels.sort_values('Hired', ascending=False)

print('=== HIRE RATE BY SOURCE CHANNEL ===')
print(channels[['source_channel','Candidates','Hired','Hire Rate','Response Rate']].to_string(index=False))

print('\n⚠️  INSIGHT: Referrals have the LOWEST hire rate AND lowest response rate')
print('⚠️  INSIGHT: Github and Inbound outperform all other channels')

=== HIRE RATE BY SOURCE CHANNEL ===
   source_channel  Candidates  Hired Hire Rate Response Rate
           Github          73      8     11.0%         68.5%
       Comunidade          89      7      7.9%         67.4%
          Inbound          69      7     10.1%         79.7%
          Hunting          75      7      9.3%         64.0%
         LinkedIn          81      7      8.6%         63.0%
Banco de Talentos          68      6      8.8%         75.0%
           Evento          82      4      4.9%         67.1%
        Indicação          64      3      4.7%         59.4%

⚠️  INSIGHT: Referrals have the LOWEST hire rate AND lowest response rate
⚠️  INSIGHT: Github and Inbound outperform all other channels


## 3. What Signals Indicate a Higher Chance of Advancing?

Score thresholds, stage duration, and recruiter notes as predictors.

In [27]:
# Score comparison: hired vs not hired
scores = df.groupby('hired')[['technical_test_score','behavior_score','manager_score']].mean().round(1)
scores.index = ['Not Hired', 'Hired']

print('=== AVERAGE SCORES: HIRED vs NOT HIRED (0-100 scale) ===')
print(scores)

# Soft threshold analysis
print('\n=== OFFER RATE BY SCORE THRESHOLD ===')
tested = df[df['test_taken'] == True]
for threshold in [70, 75, 80]:
    passed = tested[
        (tested['technical_test_score'] >= threshold) &
        (tested['behavior_score'] >= threshold) &
        (tested['manager_score'] >= threshold)
    ]
    offer_rate = (passed['offer_sent'].sum() / len(passed) * 100).round(1)
    print(f'Score >= {threshold} in all 3 dimensions: {len(passed)} candidates | {offer_rate}% offer rate')

# Hard floor
low_tech = df[df['technical_test_score'] < 70]
print(f'\nTechnical score < 70: {len(low_tech)} candidates | {low_tech["hired"].sum()} hired | 0% hire rate (hard floor)')

=== AVERAGE SCORES: HIRED vs NOT HIRED (0-100 scale) ===
           technical_test_score  behavior_score  manager_score
Not Hired                  72.9            74.0           73.1
Hired                      82.4            80.1           80.9

=== OFFER RATE BY SCORE THRESHOLD ===
Score >= 70 in all 3 dimensions: 78 candidates | 75.6% offer rate
Score >= 75 in all 3 dimensions: 46 candidates | 82.6% offer rate
Score >= 80 in all 3 dimensions: 15 candidates | 66.7% offer rate

Technical score < 70: 221 candidates | 0 hired | 0% hire rate (hard floor)


In [28]:
# Stage duration signal
screened = df[df['screening_pass'] == True]
duration = screened.groupby('hired')['stage_duration_days'].mean().round(1)
duration.index = ['Not Hired', 'Hired']

print('=== STAGE DURATION (among candidates who passed screening) ===')
print(duration)
print('\nINSIGHT: Hired candidates stay in process nearly 2x longer — sustained engagement is a positive signal')

=== STAGE DURATION (among candidates who passed screening) ===
Not Hired    16.9
Hired        32.7
Name: stage_duration_days, dtype: float64

INSIGHT: Hired candidates stay in process nearly 2x longer — sustained engagement is a positive signal


In [29]:
# Recruiter notes analysis
notes = df.groupby('recruiter_notes').agg(
    Total=('candidate_id', 'count'),
    Hired=('hired', 'sum')
).reset_index()
notes['Hire Rate'] = (notes['Hired'] / notes['Total'] * 100).round(1)
notes = notes.sort_values('Hire Rate', ascending=False)

print('=== HIRE RATE BY RECRUITER NOTE ===')
print(notes.to_string(index=False))

print('\nINSIGHT: "High salary expectations" = 14% hire rate (highest)')
print('INSIGHT: "Strong role fit" = 1.7% hire rate (lowest) — counterintuitive')

=== HIRE RATE BY RECRUITER NOTE ===
                                   recruiter_notes  Total  Hired  Hire Rate
  Perfil com forte aderência e processo acelerado.      1      1      100.0
Experiência sólida, mas expectativa salarial alta.     57      8       14.0
                Boa comunicação e resposta rápida.     52      6       11.5
            Pontos fortes em execução e autonomia.     64      7       10.9
                    Perfil técnico acima da média.     59      6       10.2
         Baixa responsividade nas etapas iniciais.     72      6        8.3
               Pediu prazo adicional para retorno.     66      5        7.6
       Demonstrou interesse claro na oportunidade.     65      4        6.2
                      Feedback positivo do gestor.     52      3        5.8
                        Bom potencial de evolução.     55      2        3.6
   Apresentou aderência forte ao contexto da vaga.     58      1        1.7

INSIGHT: "High salary expectations" = 14% hire rate

## 4. When Is It Worth Persisting With a Candidate?

In [30]:
# Candidates who passed interview but were not hired
passed_int = df[(df['interview1_pass'] == True) & (df['hired'] == False)]

print(f'Passed Interview 1 but not hired: {len(passed_int)}')
print('\nRejection reasons:')
print(passed_int['rejection_reason'].value_counts())

# Priority pipeline: cleared score threshold but no offer
priority = df[
    (df['test_taken'] == True) &
    (df['technical_test_score'] >= 75) &
    (df['behavior_score'] >= 75) &
    (df['manager_score'] >= 75) &
    (df['offer_sent'] == False)
]

print(f'\n=== PRIORITY PIPELINE ===')
print(f'Cleared >=75 in all dimensions but received no offer: {len(priority)}')
print('Rejection reasons for this group:')
print(priority['rejection_reason'].value_counts())
print('\nINSIGHT: All lost to external reasons (timing/headcount) — not fit problems. First call when a role opens.')

Passed Interview 1 but not hired: 185

Rejection reasons:
rejection_reason
Timing mismatch           16
Failed technical test     14
Low engagement            13
Headcount closed          13
No response               11
Salary mismatch           10
Low technical fit         10
Accepted another offer     7
Culture add mismatch       7
Name: count, dtype: int64

=== PRIORITY PIPELINE ===
Cleared >=75 in all dimensions but received no offer: 8
Rejection reasons for this group:
rejection_reason
Timing mismatch     3
Headcount closed    1
Name: count, dtype: int64

INSIGHT: All lost to external reasons (timing/headcount) — not fit problems. First call when a role opens.


## 5. When Is There a Low Probability of Conversion?

In [31]:
# No-response rate by channel
no_resp = df[df['rejection_reason'] == 'No response'].groupby('source_channel').size()
total_ch = df.groupby('source_channel').size()
no_resp_rate = (no_resp / total_ch * 100).round(1).sort_values(ascending=False)

print('=== NO-RESPONSE RATE BY CHANNEL ===')
print(no_resp_rate)

# Overall no response stats
total_no_resp = (df['rejection_reason'] == 'No response').sum()
total_logged = df['rejection_reason'].notna().sum()
total_not_hired = len(df) - df['hired'].sum()
untracked = df['rejection_reason'].isna().sum()

print(f'\nNo response: {total_no_resp} of {total_logged} logged rejections')
print(f'Untracked exits: {untracked} of {total_not_hired} non-hired candidates ({untracked/total_not_hired*100:.0f}%)')
print('\n⚠️  DATA QUALITY: 41% of exits have no rejection reason logged')

=== NO-RESPONSE RATE BY CHANNEL ===
source_channel
Indicação            42.2
LinkedIn             40.7
Comunidade           37.1
Hunting              36.0
Github               35.6
Evento               32.9
Banco de Talentos    29.4
Inbound              24.6
dtype: float64

No response: 210 of 375 logged rejections
Untracked exits: 226 of 552 non-hired candidates (41%)

⚠️  DATA QUALITY: 41% of exits have no rejection reason logged


## 6. Patterns Among Channels, Recruiters & Profiles

In [32]:
# Recruiter performance
recruiters = df.groupby('recruiter').agg(
    Total=('candidate_id', 'count'),
    Hired=('hired', 'sum')
).reset_index()
recruiters['Hire Rate'] = (recruiters['Hired'] / recruiters['Total'] * 100).round(1).astype(str) + '%'
recruiters = recruiters.sort_values('Hired', ascending=False)

print('=== RECRUITER PERFORMANCE ===')
print(recruiters.to_string(index=False))
print('\nINSIGHT: Volume ≠ performance. Gustavo: 121 candidates, 5% hire rate. Ana: 75 candidates, 13.3%.')

=== RECRUITER PERFORMANCE ===
recruiter  Total  Hired Hire Rate
    Bruno     98     15     15.3%
      Ana     75     10     13.3%
    Carla    101      9      8.9%
  Gustavo    121      6      5.0%
    Diego    103      5      4.9%
 Fernanda    103      4      3.9%

INSIGHT: Volume ≠ performance. Gustavo: 121 candidates, 5% hire rate. Ana: 75 candidates, 13.3%.


In [33]:
# Recruiter x Seniority heatmap
cross = df.groupby(['recruiter', 'seniority']).agg(
    Total=('candidate_id', 'count'),
    Hired=('hired', 'sum')
).reset_index()
cross['Hire Rate'] = (cross['Hired'] / cross['Total'] * 100).round(1)

heatmap = cross.pivot_table(index='recruiter', columns='seniority', values='Hire Rate', fill_value=0)

print('=== RECRUITER x SENIORITY HIRE RATE (%) ===')
print(heatmap)
print('\nINSIGHT: Ana = 37.5% at Staff. Bruno = consistently strong across all levels.')
print('INSIGHT: Diego and Fernanda collapse at Senior/Staff — same approach regardless of level.')

=== RECRUITER x SENIORITY HIRE RATE (%) ===
seniority  Junior  Pleno  Senior  Staff
recruiter                              
Ana           0.0   12.1    13.6   37.5
Bruno         4.8   16.1    19.4   20.0
Carla         0.0    5.4    17.6   10.0
Diego         5.9    7.5     2.9    0.0
Fernanda      6.7    2.5     5.3    0.0
Gustavo       8.3    2.1     2.6   18.2

INSIGHT: Ana = 37.5% at Staff. Bruno = consistently strong across all levels.
INSIGHT: Diego and Fernanda collapse at Senior/Staff — same approach regardless of level.


In [35]:
# Role-level patterns
roles = df.groupby('role').agg(
    Total=('candidate_id', 'count'),
    Hired=('hired', 'sum')
).reset_index()
roles['Hire Rate'] = (roles['Hired'] / roles['Total'] * 100).round(1).astype(str) + '%'
roles = roles.sort_values('Hired', ascending=False)

print('=== HIRE RATE BY ROLE ===')
print(roles.to_string(index=False))

# Best role-channel combo
role_channel = df.groupby(['role', 'source_channel']).agg(
    Total=('candidate_id', 'count'),
    Hired=('hired', 'sum')
).reset_index()
role_channel['Hire Rate'] = (role_channel['Hired'] / role_channel['Total'] * 100).round(1)
role_channel = role_channel[role_channel['Total'] >= 5].sort_values('Hire Rate', ascending=False)

print('\n=== TOP ROLE + CHANNEL COMBINATIONS ===')
print(role_channel.head(10).to_string(index=False))
print('\nINSIGHT: Analista de Recrutamento via Github = 26.7% — 3x the overall average')

=== HIRE RATE BY ROLE ===
                         role  Total  Hired Hire Rate
     Analista de Recrutamento    147     17     11.6%
                    People BP    165     14      8.5%
            Analista de Dados    162     11      6.8%
Talent Acquisition Specialist    127      7      5.5%

=== TOP ROLE + CHANNEL COMBINATIONS ===
                         role    source_channel  Total  Hired  Hire Rate
     Analista de Recrutamento            Github     15      4       26.7
     Analista de Recrutamento Banco de Talentos     19      4       21.1
     Analista de Recrutamento           Inbound     17      3       17.6
Talent Acquisition Specialist          LinkedIn     20      3       15.0
                    People BP        Comunidade     27      4       14.8
            Analista de Dados           Inbound     15      2       13.3
            Analista de Dados          LinkedIn     23      3       13.0
            Analista de Dados           Hunting     16      2       12.5
      

## 7. Data Quality Flags

Issues identified in the dataset that impact analysis reliability.

In [36]:
print('=== DATA QUALITY FLAGS ===')

# Scores outside 0-100 range
above_100 = df[
    (df['technical_test_score'] > 100) |
    (df['behavior_score'] > 100) |
    (df['manager_score'] > 100)
]
print(f'\n1. Scores outside 0-100 range: {len(above_100)} candidate(s)')
if len(above_100) > 0:
    print(above_100[['candidate_id','technical_test_score','behavior_score','manager_score','offer_sent','hired']])

# Missing rejection reasons
missing_reason = df[(df['hired'] == False) & (df['rejection_reason'].isna())]
print(f'\n2. Non-hired candidates with no rejection reason: {len(missing_reason)} ({len(missing_reason)/len(df)*100:.0f}% of total)')

# No pass/fail flags for behavior/manager stages
print(f'\n3. Behavior and manager scores: all {df["behavior_score"].notna().sum()} candidates have scores populated')
print('   But there are no pass/fail flags for these stages — impossible to measure drop-off')

print('\n=== RECOMMENDATIONS ===')
print('- Enforce score validation (min 0, max 100) at ATS level')
print('- Make rejection reason mandatory before closing a candidate')
print('- Add pass/fail flags for behavior and manager assessment stages')

=== DATA QUALITY FLAGS ===

1. Scores outside 0-100 range: 1 candidate(s)
     candidate_id  technical_test_score  behavior_score  manager_score  \
600           601                   118              12            104   

     offer_sent  hired  
600        True   True  

2. Non-hired candidates with no rejection reason: 177 (29% of total)

3. Behavior and manager scores: all 601 candidates have scores populated
   But there are no pass/fail flags for these stages — impossible to measure drop-off

=== RECOMMENDATIONS ===
- Enforce score validation (min 0, max 100) at ATS level
- Make rejection reason mandatory before closing a candidate
- Add pass/fail flags for behavior and manager assessment stages


## 8. Score Compensation Pattern

Investigating whether a holistic debrief model explains offers despite low scores.

In [37]:
offered = df[df['offer_sent'] == True]

print('=== COMPENSATION PATTERN: When one dimension is low, do others compensate? ===')

low_tech = offered[offered['technical_test_score'] < 75]
low_beh = offered[offered['behavior_score'] < 75]
low_mgr = offered[offered['manager_score'] < 75]

print(f'\nLow technical (<75): {len(low_tech)} candidates who still received offer')
print(low_tech[['technical_test_score','behavior_score','manager_score']].mean().round(1))

print(f'\nLow behavior (<75): {len(low_beh)} candidates who still received offer')
print(low_beh[['technical_test_score','behavior_score','manager_score']].mean().round(1))

print(f'\nLow manager (<75): {len(low_mgr)} candidates who still received offer')
print(low_mgr[['technical_test_score','behavior_score','manager_score']].mean().round(1))

print('\nINSIGHT: When one dimension is low, the other two score higher — consistent with holistic debrief model')

=== COMPENSATION PATTERN: When one dimension is low, do others compensate? ===

Low technical (<75): 10 candidates who still received offer
technical_test_score    72.4
behavior_score          79.6
manager_score           77.8
dtype: float64

Low behavior (<75): 10 candidates who still received offer
technical_test_score    83.0
behavior_score          65.9
manager_score           77.8
dtype: float64

Low manager (<75): 13 candidates who still received offer
technical_test_score    79.2
behavior_score          78.6
manager_score           71.3
dtype: float64

INSIGHT: When one dimension is low, the other two score higher — consistent with holistic debrief model


---
## Summary

**Key findings from this analysis:**

1. Github and Inbound outperform all other channels (11% and 10.1% hire rate)
2. Referrals underperform — 4.7% hire rate, lowest of all channels
3. No response = 210 of 375 logged rejections — outreach quality problem, not volume
4. Technical score >= 75 across all 3 dimensions → 82.6% offer rate
5. Technical score < 70 → 0% hire rate (hard floor)
6. Hired candidates stay in process nearly 2x longer than non-hired
7. Recruiter performance is level-dependent — Ana converts 37.5% of Staff candidates
8. Volume ≠ performance: highest-volume recruiter has lowest hire rate
9. Best role-channel combination: Analista de Recrutamento via Github (26.7%)
10. 41% of exits are untracked — data quality gap that limits analysis

**AI was used throughout:** Claude (Anthropic) supported data exploration, pattern recognition, hypothesis stress-testing, and narrative structuring. Every insight was validated against raw data and interpreted through the lens of recruiting practice.
